In [0]:
from scripts.utils import silver_utils
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid,logging

In [0]:
df=spark.read.table("workspace.ecommerce_bronze.df_orderitems")
logging.info('table read successfully')

In [0]:
silver_utils.check_schema(df)

In [0]:
df = silver_utils.cast_types(df,{
    "order_id" : "string" ,
    "product_id" : "string",
    "seller_id" : "string",
    "price" : "double",
    "shipping_charges" : "double"
})

In [0]:
silver_utils.null_profiling(df,"df_orderitems")
df= silver_utils.handle_nulls_drop(df,drop_cols=["order_id","product_id","seller_id","price","shipping_charges"])

In [0]:
df = silver_utils.handle_duplicates(df,["order_id","product_id","seller_id"])

In [0]:
df =silver_utils.standardize_strings(df,{
    "order_id" : lambda c : trim(c),
    "product_id" : lambda c :trim(c),
    "seller_id" : lambda c :trim(c)
})

In [0]:
df_orders = spark.read.table("workspace.ecommerce_bronze.df_orders")
df_products = spark.read.table("workspace.ecommerce_bronze.df_products")

df_with_check_order = (df.alias("orderitems").join(df_orders.select("order_id").alias("orders"),col("orderitems.order_id") == col("orders.order_id"),"left"))

df_with_check_product = (df.alias("orderitems").join(df_products.select("product_id").alias("products"),col("orderitems.product_id") == col("products.product_id"),"left"))

checks = [
    #nulls
    (
        "order_id",
        "order_id not null",
        col("order_id").isNotNull()
    ),
    (
        "product_id",
        "product_id not null",
        col("product_id").isNotNull()
    ),
    (
        "seller_id",
        "seller_id not null",
        col("seller_id").isNotNull()
    ),
    (
        "price",
        "price not null",
        col("price").isNotNull()
    ),
    (
        "shipping_charges",
        "shipping_charges not null",
        col("shipping_charges").isNotNull()
    ),

    #business rules
    (
        "price",
        "price is positive number",
        col("price") >= 0
    ),
    (
        "shipping_charges",
        "shipping_charges is positive number",
        col("shipping_charges") >= 0
    ),
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="df_orderitems",
    warn_threshold=5.0
)

#referential integrity
checks_ref_orders = [
    (
        "orders.order_id",
        "order_id exists in df_orders",
        col("orders.order_id").isNotNull()
    ),
]

checks_ref_products = [
    (
        "products.product_id",
        "product_id exists in df_products",
        col("products.product_id").isNotNull()
    ),
]

dq_ref_orders= silver_utils.build_dq_table(
    spark=spark,
    df=df_with_check_order,
    checks=checks_ref_orders,
    table_name="df_orderitems",
    warn_threshold=5.0
)

dq_ref_products= silver_utils.build_dq_table(
    spark=spark,
    df=df_with_check_product,
    checks=checks_ref_products,
    table_name="df_orderitems",
    warn_threshold=5.0
)


#combine all dq results into one table
from functools import reduce
from pyspark.sql import DataFrame

dq_final = reduce(DataFrame.union, [dq_table, dq_ref_orders, dq_ref_products])

dq_final.write.mode("overwrite").saveAsTable("workspace.ecommerce_silver.df_orderitems_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.df_orderitems_dq")
display(dq_saved)

In [0]:
df = df.withColumn("item_total_value",
    round(col("price") + col("shipping_charges"), 2)
)

tax_rate= 0.14
df=df.withColumn("payment_value_with_tax",
    round((col("price") + col("shipping_charges")) * (1 + tax_rate), 2)
)

In [0]:
df =silver_utils.add_silver_metadata(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.ecommerce_silver.df_orderitems")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.df_orderitems LIMIT 10